# AlignLab — Experiment 01: QLoRA Supervised Fine-Tuning

Train Qwen3-1.7B with 4-bit QLoRA on **Colab or Kaggle**. Save the LoRA adapter in the cloud runtime and optionally publish a compact, uniquely named experiment report to GitHub. This is a smoke test, not a code benchmark.


## 1. Install dependencies

Run this cell once, then restart the runtime if prompted. Kaggle may report unrelated Gradio/Diffusers dependency warnings.


In [ ]:
%pip install -q -U "transformers>=4.51,<5" "trl>=0.19,<0.27" \
    "peft>=0.15,<0.19" "accelerate>=1.4,<2" "datasets>=3.5,<5" \
    "bitsandbytes>=0.45" "huggingface-hub>=0.30,<1" "sentencepiece"

## 2. Check environment

In [ ]:
import json
import random
import shutil
import time
from collections import Counter
from pathlib import Path

import torch

assert torch.cuda.is_available(), (
    "Enable a GPU accelerator in Kaggle or Google Colab before running."
)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("CUDA:", torch.version.cuda)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("/kaggle/working/alignlab-qlora")
elif Path("/content").exists():
    OUTPUT_DIR = Path("/content/alignlab-qlora")
else:
    OUTPUT_DIR = Path("./alignlab-qlora")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_DIR)

## 3. Configuration

The default 100 training examples and 30 steps are intentionally small for a first run. Increase them only after the smoke test succeeds. The model is **Qwen3-1.7B**, and we disable its reasoning mode for this straightforward instruction-following experiment.

In [ ]:
MODEL_ID = "Qwen/Qwen3-1.7B"
MAX_TRAIN_EXAMPLES = 100
MAX_EVAL_EXAMPLES = 20
MAX_STEPS = 30
MAX_LENGTH = 512
LEARNING_RATE = 2e-4
LORA_R = 16
LORA_ALPHA = 32


## 4. Load tokenizer and 4-bit base model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16,
)
model.config.use_cache = False
print("Loaded:", MODEL_ID)
print("4-bit quantization:", model.is_loaded_in_4bit)

## 5. Load and prepare data

We use `yahma/alpaca-cleaned` as a convenient public instructional dataset. Review the dataset's provenance and licensing before redistributing derivatives. We shuffle and split the selected subset so validation examples are not used for training. This dataset is **not** suitable for making code-generation improvement claims.

In [ ]:
from datasets import load_dataset

raw = load_dataset("yahma/alpaca-cleaned", split="train")
raw = raw.shuffle(seed=SEED).select(range(min(len(raw), MAX_TRAIN_EXAMPLES + MAX_EVAL_EXAMPLES)))

def to_text(example):
    user_content = example["instruction"]
    if example.get("input", "").strip():
        user_content += "\n\nInput:\n" + example["input"]
    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, enable_thinking=False)}

processed = raw.map(to_text, remove_columns=raw.column_names)
train_count = min(MAX_TRAIN_EXAMPLES, len(processed) - 1)
train_dataset = processed.select(range(train_count))
eval_dataset = processed.select(range(train_count, len(processed)))
print("Train:", len(train_dataset), "Eval:", len(eval_dataset))
print(train_dataset[0]["text"][:700])

## 6. Capture a baseline response

This is a qualitative comparison only; use held-out metrics for real model comparisons.

In [ ]:
PROMPT = (
    "Explain the difference between supervised fine-tuning and "
    "retrieval-augmented generation in three bullet points."
)


def generate_response(active_model, prompt):
    chat = [{"role": "user", "content": prompt}]
    rendered = tokenizer.apply_chat_template(
        chat,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(rendered, return_tensors="pt").to("cuda")
    active_model.eval()
    with torch.inference_mode():
        ids = active_model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(
        ids[0, inputs.input_ids.shape[-1]:],
        skip_special_tokens=True,
    )


baseline_response = generate_response(model, PROMPT)
print("BASELINE:\n", baseline_response)


## 7. Configure LoRA and SFT training

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    dataset_text_field="text",
    max_length=MAX_LENGTH,
    packing=False,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=15,
    save_strategy="steps",
    save_steps=15,
    save_total_limit=2,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

# AMP gradient scaling requires compatible trainable gradient dtypes.
for param in trainer.model.parameters():
    if param.requires_grad and param.dtype != torch.float32:
        param.data = param.data.float()

trainable_dtypes = Counter(
    str(param.dtype)
    for param in trainer.model.parameters()
    if param.requires_grad
)
print("Trainable parameter dtypes:", trainable_dtypes)
print("FP16:", training_args.fp16, "BF16:", training_args.bf16)
trainer.model.print_trainable_parameters()

## 8. Train

On a free T4, training time varies with GPU availability and current load. If you get an out-of-memory error, restart the kernel, lower `MAX_LENGTH` to 256, and retry from the top.

In [ ]:
start = time.perf_counter()
train_result = trainer.train()
train_seconds = time.perf_counter() - start
print(f"Training time: {train_seconds:.1f} seconds")
print(train_result.metrics)

## 9. Save adapter and evaluate

In [ ]:
adapter_dir = OUTPUT_DIR / "adapter"
trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
eval_metrics = trainer.evaluate()
print("Evaluation:", eval_metrics)

# Generation can run on the trainer's PEFT-wrapped model.
trainer.model.config.use_cache = True
finetuned_response = generate_response(trainer.model, PROMPT)
print("BASELINE:\n", baseline_response)
print("\nFINE-TUNED:\n", finetuned_response)

report = {
    "model": MODEL_ID,
    "dataset": "yahma/alpaca-cleaned",
    "seed": SEED,
    "train_examples": len(train_dataset),
    "eval_examples": len(eval_dataset),
    "steps": MAX_STEPS,
    "train_seconds": train_seconds,
    "train_metrics": train_result.metrics,
    "eval_metrics": eval_metrics,
    "prompt": PROMPT,
    "baseline_response": baseline_response,
    "finetuned_response": finetuned_response,
}
with (OUTPUT_DIR / "experiment_report.json").open("w") as f:
    json.dump(report, f, indent=2, default=str)
print("Saved to", OUTPUT_DIR)

## 10. Package local outputs

The ZIP contains the adapter and experiment report. Download it from Kaggle outputs or Colab Files before the runtime expires. Large model artifacts are **not** pushed to GitHub.


In [ ]:
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Adapter + report archive:", archive)

## Next steps

1. Move this notebook into your GitHub repository under `notebooks/`.
2. Replace the general instruction dataset with a carefully curated **code-generation SFT dataset** and ensure HumanEval is held out.
3. Add a safe, isolated evaluation harness before reporting pass@1 (never run untrusted generated code directly in the notebook).
4. Run full evaluation on base and fine-tuned checkpoints with identical generation settings.
5. Prepare preference pairs and run DPO as a separate experiment.

**Reproducibility note:** Pin exact dependency versions after your first successful Kaggle run; the version ranges above are intended to get the notebook started and can change over time.

## 11. Optional: publish metrics to GitHub (Colab **and** Kaggle)

Set a fine-grained GitHub token with **Contents: Read and write** access to **only** the AlignLab repository.

- **Colab:** Secrets panel → `GITHUB_TOKEN` → enable notebook access.
- **Kaggle:** Add `GITHUB_TOKEN` under notebook **Add-ons → Secrets**, then allow the notebook to use it.

The cell publishes **only** `experiment_report.json` to `results/qlora/<unique-run-id>.json` on the configured branch. It does **not** upload checkpoints, ZIP files, or credentials. If the secret is absent, publishing is skipped. Avoid putting sensitive/private training data in the report. GitHub branch protection may prevent direct writes.

In [ ]:
import base64
import datetime as dt
import os
import urllib.error
import urllib.parse
import urllib.request
import uuid

GITHUB_REPO = "hamedhani/alignlab"  # Change if your GitHub owner differs
GITHUB_BRANCH = "main"
PUBLISH_TO_GITHUB = True


def read_github_token():
    if Path("/content").exists():
        try:
            from google.colab import userdata

            return userdata.get("GITHUB_TOKEN")
        except Exception as exc:
            print("Colab GitHub secret unavailable:", type(exc).__name__)
    if Path("/kaggle/working").exists():
        try:
            from kaggle_secrets import UserSecretsClient

            return UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception as exc:
            print("Kaggle GitHub secret unavailable:", type(exc).__name__)
    return os.environ.get("GITHUB_TOKEN")


def publish_report_to_github(report_file, token):
    stamp = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_id = f"qlora-{stamp}-{uuid.uuid4().hex[:8]}"
    destination = f"results/qlora/{run_id}.json"
    repo_path = urllib.parse.quote(GITHUB_REPO, safe="/")
    file_path = urllib.parse.quote(destination, safe="/")
    url = f"https://api.github.com/repos/{repo_path}/contents/{file_path}"
    payload = {
        "message": f"results: add QLoRA experiment {run_id}",
        "content": base64.b64encode(report_file.read_bytes()).decode("ascii"),
        "branch": GITHUB_BRANCH,
    }
    request = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        method="PUT",
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "Content-Type": "application/json",
        },
    )
    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            if response.status != 201:
                raise RuntimeError(f"Unexpected GitHub response: {response.status}")
    except urllib.error.HTTPError as exc:
        raise RuntimeError(
            f"GitHub report upload failed (HTTP {exc.code}). "
            "Check token access, repository name, and branch permissions."
        ) from None
    return f"https://github.com/{GITHUB_REPO}/blob/{GITHUB_BRANCH}/{destination}"


if PUBLISH_TO_GITHUB:
    github_token = read_github_token()
    if github_token:
        report_file = OUTPUT_DIR / "experiment_report.json"
        if not report_file.is_file():
            raise FileNotFoundError(f"Missing training report: {report_file}")
        report_url = publish_report_to_github(report_file, github_token)
        print("Published experiment report:", report_url)
    else:
        print("No GITHUB_TOKEN secret available; skipped GitHub publishing.")
else:
    print("GitHub publishing disabled.")

## 12. Next steps

1. Inspect `experiment_report.json` and the adapter ZIP.
2. Keep the LoRA adapter on Hugging Face Hub or download it; do not commit large weights to GitHub.
3. Replace the general instruction dataset with a curated code-training dataset, holding HumanEval out of training.
4. Compare base, SFT, and later DPO checkpoints with identical evaluation settings.

**Security:** Never paste tokens into cells or commit executed notebook outputs containing secrets. Use each platform's secret manager. If GitHub publishing fails, the local results remain available.